# Lab 26: Linear Algebra and Probability

Independent-study notebook with worked solutions and similar practice questions.

## 1. Random vectors, mean vectors, and covariance matrices

For data matrix `X` with rows as observations, the sample mean is `X.mean(axis=0)` and the sample covariance is `np.cov(X, rowvar=False)`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)

X = np.array([[2.0, 1.0], [4.0, 3.0], [6.0, 5.0], [8.0, 7.0]])
print("mean:", X.mean(axis=0))
print("covariance:
", np.cov(X, rowvar=False))

### Question 1
Compute the mean and covariance for the following data set. Then interpret the sign of the off-diagonal covariance.

In [ ]:
Y = np.array([[1, 2], [2, 4], [3, 6], [4, 8], [5, 10]], dtype=float)
print("mean:", Y.mean(axis=0))
print("covariance:
", np.cov(Y, rowvar=False))
# Answer: the variables move together perfectly, so covariance is positive and rank is 1.

### Similar practice 1
Change the second coordinate to decrease as the first coordinate increases. What happens to the off-diagonal covariance?

In [ ]:
Y2 = np.array([[1, 10], [2, 8], [3, 6], [4, 4], [5, 2]], dtype=float)
print("mean:", Y2.mean(axis=0))
print("covariance:
", np.cov(Y2, rowvar=False))
# Answer: the off-diagonal covariance is negative.

## 2. Covariance eigenvectors and principal directions

In [ ]:
Sigma = np.array([[4.0, 3.0], [3.0, 4.0]])
vals, vecs = np.linalg.eigh(Sigma)
idx = vals.argsort()[::-1]
vals, vecs = vals[idx], vecs[:, idx]
print("eigenvalues:", vals)
print("eigenvectors:
", vecs)

In [ ]:
theta = np.linspace(0, 2*np.pi, 300)
circle = np.vstack([np.cos(theta), np.sin(theta)])
ellipse = vecs @ np.diag(np.sqrt(vals)) @ circle
plt.figure(figsize=(5,5))
plt.plot(ellipse[0], ellipse[1])
plt.scatter([0], [0])
plt.gca().set_aspect('equal', adjustable='box')
plt.title('Covariance ellipse')
plt.show()

### Question 2
Verify numerically that the maximum of $u^T\Sigma u$ over unit vectors occurs near the first eigenvector.

In [ ]:
angles = np.linspace(0, 2*np.pi, 1000)
variances = []
for t in angles:
    u = np.array([np.cos(t), np.sin(t)])
    variances.append(u @ Sigma @ u)
variances = np.array(variances)
print("max variance:", variances.max())
print("largest eigenvalue:", vals[0])
print("angle at max:", angles[variances.argmax()])
# Answer: the maximum matches the largest eigenvalue.

## 3. Affine transformations of random vectors

In [ ]:
A = np.array([[1.0, 1.0], [0.0, 1.0]])
mu = np.array([1.0, -1.0])
new_mu = A @ mu
new_cov = A @ Sigma @ A.T
print("new mean:", new_mu)
print("new covariance:
", new_cov)

### Question 3
Let $A=egin{bmatrix}2&0\0&1\end{bmatrix}$. Compute the transformed covariance.

In [ ]:
A2 = np.array([[2.0, 0.0], [0.0, 1.0]])
print(A2 @ Sigma @ A2.T)
# Answer: scaling the first coordinate by 2 multiplies first variance by 4 and covariance terms involving that coordinate by 2.

## 4. Gaussian simulation and whitening

In [ ]:
np.random.seed(0)
mu = np.array([1.0, -1.0])
X = np.random.multivariate_normal(mu, Sigma, size=5000)
print("sample covariance before whitening:
", np.cov(X, rowvar=False))

sample_mean = X.mean(axis=0)
sample_cov = np.cov(X, rowvar=False)
lam, Q = np.linalg.eigh(sample_cov)
idx = lam.argsort()[::-1]
lam, Q = lam[idx], Q[:, idx]
W = np.diag(1/np.sqrt(lam)) @ Q.T
Z = (W @ (X - sample_mean).T).T
print("sample covariance after whitening:
", np.cov(Z, rowvar=False))

### Question 4
Explain what whitening does geometrically.

**Answer.** Whitening first rotates into the principal-coordinate system and then rescales each coordinate by its standard deviation. The covariance becomes approximately the identity matrix.

## 5. High-dimensional random directions

In [ ]:
np.random.seed(1)
for d in [10, 50, 100, 500, 1000]:
    trials = 500
    dots = []
    for _ in range(trials):
        u = np.random.randn(d); u /= np.linalg.norm(u)
        v = np.random.randn(d); v /= np.linalg.norm(v)
        dots.append(u @ v)
    dots = np.array(dots)
    print(d, "RMS dot:", np.sqrt(np.mean(dots**2)), "theory:", 1/np.sqrt(d))

### Similar practice 2
Try dimensions $d=2000$ and $d=5000$. What happens to typical dot products?

In [ ]:
for d in [2000, 5000]:
    trials = 100
    dots = []
    for _ in range(trials):
        u = np.random.randn(d); u /= np.linalg.norm(u)
        v = np.random.randn(d); v /= np.linalg.norm(v)
        dots.append(u @ v)
    print(d, np.sqrt(np.mean(np.array(dots)**2)), 1/np.sqrt(d))
# Answer: typical dot products shrink toward 0.

## 6. Sample covariance rank limitation

In [ ]:
d = 1000
n = 50
X = np.random.randn(d, n)
Xc = X - X.mean(axis=1, keepdims=True)
S = (1/(n-1)) * Xc @ Xc.T
print("rank of centered data:", np.linalg.matrix_rank(Xc))
print("rank of sample covariance:", np.linalg.matrix_rank(S, tol=1e-8))
print("largest possible rank:", n-1)

### Question 5
Why is the sample covariance singular when $d>n-1$?

**Answer.** Since the centered columns sum to zero, $\operatorname{rank}(X_c)\le n-1$. Therefore $\operatorname{rank}(X_cX_c^T)\le n-1<d$, so the $d	imes d$ covariance matrix cannot be invertible.

## 7. Random matrices

In [ ]:
np.random.seed(2)
d, n = 100, 300
X = np.random.randn(d, n)
S = (1/n) * X @ X.T
eigs = np.linalg.eigvalsh(S)
print("min eigenvalue:", eigs[0])
print("max eigenvalue:", eigs[-1])
print("mean eigenvalue:", eigs.mean())
plt.figure(figsize=(6,4))
plt.hist(eigs, bins=30)
plt.title('Eigenvalues of a random sample covariance matrix')
plt.xlabel('eigenvalue')
plt.ylabel('count')
plt.show()

## Final reflection
Write a short paragraph explaining how covariance matrices connect probability, geometry, and PCA.